# CLIPSeg + SAM3Tracker 하이브리드 — 마스크 정제 실측

**스파이크/검증용 코드입니다. 프로덕션 코드가 아닙니다.**

설계: `.claude/plans/ai/007-clipseg-sam3tracker-vs-sam3-solo-comparison.md`

## 왜 이 노트북이 필요한가

노션 LAB-46(`[AI] 시스템 아키텍처 — CLIPSeg+SAM Tracker vs SAM3 단독`)의 종합 의견은
**"지금 시점 증거로는 CLIPSeg+Tracker 가 더 안전하다"** 였지만, 그 결론에 구멍을 두 개 남겼다.

1. 라벨링된 검증셋이 없어 전부 육안 판단이었다 — 정량 비교가 아니다.
2. **CLIPSeg+Tracker 조합을 실제로 돌려본 적이 없다.** `sam3_qc_test.ipynb` §9(SAM3 PCS 자체
   마스크를 Tracker 로 정제)는 설계만 하고 실행되지 않았고, **CLIPSeg 박스를 Tracker 에 먹이는
   조합은 코드로 존재하지 않았다.**

이 노트북이 **구멍 2번**을 메운다. 구멍 1번(라벨링 검증셋)은 스코프 밖이다 — 여전히
`sample_images/` 실제 상품 사진을 보는 수준이지만, "돌려본 적 없다"를 해소하는 게 목적이다.

## 이 노트북이 답하려는 질문

| # | 질문 | 섹션 |
| --- | --- | --- |
| 1 | CLIPSeg 와 SAM3Tracker 를 **동시에 적재**할 수 있는가 (LAB-46 §4 추정 1.85GB 가 맞는가) | §3 |
| 2 | CLIPSeg 박스를 Tracker 에 넣으면 마스크가 실제로 **타이트해지는가** | §4, §5 |
| 3 | class-agnostic 함정 — 정제가 아니라 **대상이 바뀐** 사례가 얼마나 나오는가 | §5 |
| 4 | SAM3 단독과 비교했을 때 같은 곳을 짚는가 | §6 |

## ⚠️ 개념상 반드시 알고 읽어야 하는 것

**SAM3Tracker 는 탐지·분류를 하지 않는다.** `Sam3TrackerModel` 은 텍스트 프롬프트를 받지 못하는
PVS(포인트/박스/마스크 프롬프트 전용) class-agnostic 세그멘터다. 즉 이 하이브리드에서

- **탐지·분류·개수는 CLIPSeg 가 100% 결정한다** — Tracker 를 붙여도 인스턴스 개수·유형은
  `clipseg_qc_test.ipynb` 와 **똑같이 나온다.**
- Tracker 가 바꾸는 것은 **각 인스턴스의 마스크 경계뿐**이다.

그래서 SAM3 단독과의 비교축은 "몇 개 잡았나"가 아니라 **"마스크 품질"** 이다. 개수·판정 비교는
이미 `clipseg_qc_test.ipynb` §7 이 하고 있고, 이 노트북이 새로 답하는 건 경계 품질이다.

## 비교가 성립하려면 — 공유 모듈

프롬프트·임계치·필터를 노트북마다 복사하면 한쪽만 고쳐져 조용히 어긋난다. 공유분은 전부
**`validation/QUAL/qc_shared.py`** 한 곳에 있다(세 노트북이 같은 것을 본다).

- 이미지: `qc_shared.SHARED_IMAGE_DIR` (= `sam3-condition-check/sample_images/`)
- 프롬프트: `DEFECT_PROMPTS` / `BENIGN_PROMPTS` / `GARMENT_PROMPT`
- 후처리: `probs_to_instances()` / `filter_by_garment()` / `suppress_benign()` / `decide_verdict()`
- 정제 비교용: `mask_iou()`

## 실행 순서

셀을 위에서 아래로 순서대로 실행한다. **`sam3_qc_test.ipynb` 를 먼저 끝까지 돌려**
`sam3-condition-check/outputs/*_qc.json` 을 만들어 두면 §6 비교가 켜진다(없으면 그 섹션만 건너뜀).

> `facebook/sam3` 는 **gated repo** 라 HF 약관 동의·승인이 필요하다. CLIPSeg 는 게이트가 없다.

## 0. 의존성 · 환경

SAM3/CLIPSeg 노트북과 **같은 conda 환경(`qual`)** 을 쓴다. 이 노트북은 두 모델을 동시에
올리므로 VRAM 여유가 가장 빡빡한 케이스다 — RTX 4070 8GB 기준으로 설계했다.

In [ ]:
%pip install -q -r ../sam3-condition-check/requirements-sam3.txt

In [ ]:
import torch

print(f"torch          : {torch.__version__}")
print(f"cuda available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    _props = torch.cuda.get_device_properties(0)
    print(f"gpu            : {_props.name}")
    print(f"total VRAM     : {_props.total_memory / 1024**3:.2f} GB")
else:
    print("[주의] CPU 모드입니다. Tracker 추론이 매우 느리고 VRAM 실측(§3)이 무의미해집니다.")

## 0-1. 설정 — 프롬프트·임계치는 `qc_shared` 에서 온다

여기서 프롬프트를 새로 정의하지 않는다. 바꾸려면 `validation/QUAL/qc_shared.py` 의
`ACTIVE_PROMPT_SET` 을 고친다 — 그러면 세 노트북이 함께 바뀐다.

### 이 노트북만의 상수

정제 판정 임계치(`REFINE_IOU_MIN` / `REFINE_PX_RATIO_MAX` / `REFINE_TIGHTEN_MAX`)는
**근거 없는 초기값이다.** §5 결과를 보고 조정해야 하며, 이 값에 따라 "정제됨/대상변경"
분류가 바뀐다는 점을 결론에 반드시 명시할 것.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "..")
from qc_shared import (  # noqa: E402
    ACTIVE_PROMPT_SET,
    BACKEND_THRESHOLDS,
    BENIGN_CONTAINMENT,
    BENIGN_PROMPTS,
    DEFECT_PROMPTS,
    GARMENT_OVERLAP_MIN,
    GARMENT_PROMPT,
    KOREAN_LABELS,
    RESTRICT_TO_GARMENT,
    SAM3_OUTPUT_DIR,
    SHARED_IMAGE_DIR,
    SUPPRESS_BENIGN,
    VERDICT_THRESHOLDS,
    decide_verdict,
    draw_numbered_boxes,
    filter_by_garment,
    list_images,
    mask_iou,
    pick_largest,
    probs_to_instances,
    renumber,
    suppress_benign,
    to_serializable,
    use_korean_font,
)

# --- 모델 ---
CLIPSEG_MODEL_ID = "CIDAS/clipseg-rd64-refined"
CLIPSEG_INPUT_SIZE = 224  # worker/models/clipseg/segmenter.py 와 같은 값
SAM3_MODEL_ID = "facebook/sam3"  # Tracker 도 같은 체크포인트를 쓴다

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# CLIPSeg 는 fp32(~150MB 라 fp16 이득이 거의 없다), Tracker 는 fp16(~1.7GB 라 이득이 크다).
# 서로 다른 dtype 이라 Tracker 입력만 캐스팅해 준다(_to_tracker_device 참고).
TRACKER_DTYPE = torch.float16

OUTPUT_DIR = Path("./outputs")

# --- 임계치 (백엔드별로 분리 — qc_shared.BACKEND_THRESHOLDS 주석 참고) ---
PROB_THRESHOLD = BACKEND_THRESHOLDS["clipseg"]["prob"]
GARMENT_PROB_THRESHOLD = BACKEND_THRESHOLDS["clipseg"]["garment_prob"]

# --- 크롭 실험 토글 — CLIPSeg 노트북과 같은 이유로 여기 지역 변수로 둔다 ---
# ⚠️ 실측 상품 사진 8장에서 True(크롭)가 6장 회귀를 냈다. 기본은 False.
CROP_BEFORE_DETECT = False

# --- 정제 실험 설정 (이 노트북 고유) ---
# Tracker 는 이미지 1장의 박스를 한 번에 받지만, 박스가 많으면 VRAM 이 튄다. 8GB 에서 추론
# 중간에 OOM 이 나면 그 이미지 결과를 통째로 잃으므로 나눠 호출한다(대신 vision 인코더가
# chunk 마다 다시 돈다 — 스파이크라 속도보다 안전을 택했다).
REFINE_BATCH_SIZE = 8

# ⚠️ 아래 3개는 근거 없는 초기값이다. §5 실측을 보고 조정할 것.
# 정제라면 "같은 대상의 경계만" 달라져야 하므로 IoU 가 높아야 정상이다.
REFINE_IOU_MIN = 0.35  # 이 미만이면 같은 대상이라고 보기 어렵다
# 마스크가 폭증하면 하자가 아니라 원단 표면 전체를 잡은 것일 수 있다(class-agnostic 함정).
REFINE_PX_RATIO_MAX = 2.0
# 픽셀이 이 비율 미만으로 줄었으면 "타이트해졌다"고 본다 — 정제로 얻는 이득의 실체다.
REFINE_TIGHTEN_MAX = 0.8

TOPK_VISUALIZE = 4  # §5-1 에서 확대해 나란히 볼 상위 인스턴스 개수

print(f"프롬프트 세트 : {ACTIVE_PROMPT_SET}  (SAM3·CLIPSeg 노트북과 공유)")
for _t, _p in DEFECT_PROMPTS.items():
    print(f"  {_t:<8}: {_p}")
print()
print(f"CLIPSeg       : {CLIPSEG_MODEL_ID}  fp32  prob>{PROB_THRESHOLD}")
print(f"Tracker       : {SAM3_MODEL_ID}  {TRACKER_DTYPE}  (탐지 아님 — 마스크 정제 전용)")
print(f"의류 게이트   : {RESTRICT_TO_GARMENT}  (존재확인 임계={GARMENT_PROB_THRESHOLD})")
print(f"  CROP_BEFORE_DETECT: {CROP_BEFORE_DETECT}  (실측 회귀로 기본 False)")
print(f"양성 억제     : {SUPPRESS_BENIGN}  (잠김 비율 {BENIGN_CONTAINMENT})")
print()
print("정제 판정 임계 (⚠️ 근거 없는 초기값):")
print(f"  IoU 최소       : {REFINE_IOU_MIN}   (미만 -> 대상변경 의심)")
print(f"  픽셀비 최대    : {REFINE_PX_RATIO_MAX}    (초과 -> 대상변경 의심)")
print(f"  타이트 판정    : {REFINE_TIGHTEN_MAX}    (미만 -> 경계가 좁아짐 = 정제 이득)")
print(f"이미지 폴더   : {SHARED_IMAGE_DIR}")

## 0-2. CLIPSeg 로드

**로드 순서가 중요하다.** CLIPSeg 를 먼저 올려 §1~§2 에서 인스턴스를 **전부 뽑아 둔 뒤**
§3 에서 Tracker 를 올린다. 그래야

1. Tracker 로드 시점의 VRAM 델타가 "CLIPSeg 가 이미 올라간 상태에서 추가로 드는 양"이 되고,
2. 만약 동시 적재가 OOM 이면 CLIPSeg 를 내리고 재시도해도 **이미 계산된 인스턴스를 잃지 않는다**
   (마스크는 numpy 라 CPU 에 있다).

`sam3_qc_test.ipynb` §9 는 PCS 를 내리는 폴백에서 이후 셀이 깨지는 구조였는데, 여기서는
탐지가 먼저 끝나므로 그 문제가 없다.

In [ ]:
from transformers import CLIPSegForImageSegmentation, CLIPSegProcessor

if DEVICE == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


def vram_report(tag):
    """(allocated_MB, peak_MB) 를 찍고 반환. empty_cache 후 재므로 캐시 블록은 제외된다."""
    if DEVICE != "cuda":
        print(f"{tag:<28} (CPU 모드 — VRAM 측정 없음)")
        return (0.0, 0.0)
    torch.cuda.empty_cache()
    alloc = torch.cuda.memory_allocated() / 1024**2
    peak = torch.cuda.max_memory_allocated() / 1024**2
    print(f"{tag:<28} allocated={alloc:9.1f} MB ({alloc / 1024:.2f} GB)   peak={peak:9.1f} MB")
    return (alloc, peak)


try:
    processor = CLIPSegProcessor.from_pretrained(CLIPSEG_MODEL_ID)
    clipseg_model = CLIPSegForImageSegmentation.from_pretrained(CLIPSEG_MODEL_ID).to(DEVICE).eval()
    print(f"로드 완료: {CLIPSEG_MODEL_ID}  device={DEVICE}")
except torch.cuda.OutOfMemoryError as exc:
    print(f"[CUDA OOM] {exc}")
    print("  다른 노트북 커널이 GPU 를 점유하고 있을 수 있습니다 — 그쪽 커널을 종료하세요.")
    raise
except Exception as exc:
    print(f"[모델 로드 실패] {type(exc).__name__}: {exc}")
    raise

print()
VRAM_CLIPSEG_ONLY, _ = vram_report("CLIPSeg 단독 적재 후")

## 0-3. 이미지 로드 — 세 노트북이 같은 폴더를 본다

`qc_shared.list_images()` 가 `sam3-condition-check/sample_images/` 를 하위 폴더까지 훑는다.
같은 목록·같은 순서여야 비교가 성립한다.

In [ ]:
from PIL import Image

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_PATHS = list_images()

if not IMAGE_PATHS:
    raise FileNotFoundError(
        f"{SHARED_IMAGE_DIR} 에 이미지가 없습니다. "
        "sam3_qc_test.ipynb 를 먼저 실행하거나 이미지를 직접 넣어 주세요."
    )


def load_image(path):
    return Image.open(path).convert("RGB")


print(f"이미지 {len(IMAGE_PATHS)}장:")
for _i, _p in enumerate(IMAGE_PATHS):
    print(f"  [{_i}] {_p.relative_to(SHARED_IMAGE_DIR)}")

## 1. CLIPSeg 탐지 — 하이브리드의 1단(탐지·분류)

`clipseg_qc_test.ipynb` §1~§3 과 **완전히 같은 로직**이다. 후처리는 `qc_shared` 공유 함수를
그대로 쓰고, 확률맵→인스턴스 변환도 공유 함수(`probs_to_instances`)를 쓴다 — 그래야 여기
결과가 CLIPSeg 단독 노트북과 일치하고, 차이가 나면 "Tracker 때문"이라고 말할 수 있다.

`processor(...)` 통합 호출 대신 **`tokenizer` / `image_processor` 분리 호출**을 쓴다. 통합
호출은 텍스트/이미지 kwarg 가 섞여 에러가 난다 — LAB-12 `verify_clipseg.py` 와 프로덕션
`segmenter.py` 가 쓰는 것과 같은 우회다.

In [ ]:
import numpy as np


def clipseg_probs(pil_image, prompts):
    """프롬프트 리스트 -> (n, H, W) 확률맵 (원본 해상도)."""
    text_inputs = processor.tokenizer(list(prompts), return_tensors="pt", padding=True)
    image_inputs = processor.image_processor(
        [pil_image] * len(prompts),
        size={"height": CLIPSEG_INPUT_SIZE, "width": CLIPSEG_INPUT_SIZE},
        return_tensors="pt",
    )
    inputs = {**text_inputs, "pixel_values": image_inputs["pixel_values"]}
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = clipseg_model(**inputs)

    logits = outputs.logits
    if logits.ndim == 2:  # 프롬프트가 1개면 배치 차원이 사라진다
        logits = logits.unsqueeze(0)

    probs = torch.sigmoid(logits).unsqueeze(1)  # (n, 1, h, w)
    probs = torch.nn.functional.interpolate(
        probs,
        size=(pil_image.height, pil_image.width),
        mode="bilinear",
        align_corners=False,
    )
    return probs[:, 0].detach().cpu().numpy()


def detect_all(pil_image, prompts=DEFECT_PROMPTS, threshold=PROB_THRESHOLD):
    """하자 프롬프트 전체 -> 인스턴스 병합 리스트."""
    tags = list(prompts)
    maps = clipseg_probs(pil_image, [prompts[t] for t in tags])
    merged = []
    for tag, prob_map in zip(tags, maps):
        merged.extend(probs_to_instances(prob_map, tag, prompts[tag], threshold))
    return renumber(merged)


def benign_instances(pil_image, threshold=PROB_THRESHOLD):
    """주름·접힘·그림자·솔기·프린트 등 정상 영역 인스턴스(억제용)."""
    tags = list(BENIGN_PROMPTS)
    maps = clipseg_probs(pil_image, [BENIGN_PROMPTS[t] for t in tags])
    found = []
    for tag, prob_map in zip(tags, maps):
        found.extend(probs_to_instances(prob_map, tag, BENIGN_PROMPTS[tag], threshold))
    return found


def garment_mask_full(pil_image, threshold=GARMENT_PROB_THRESHOLD):
    """원본 전체 이미지 기준 의류 마스크(가장 큰 인스턴스). 못 찾으면 None."""
    prob_map = clipseg_probs(pil_image, [GARMENT_PROMPT])[0]
    picked = pick_largest(probs_to_instances(prob_map, "GARMENT", GARMENT_PROMPT, threshold))
    return picked["mask"] if picked is not None else None


def inspect_image(pil_image):
    """하자 탐지 -> 의류 게이트(사후 필터) -> 양성 억제. (최종, 단계별 기록) 반환.

    CROP_BEFORE_DETECT 는 실측 회귀로 기본 False 이므로, 여기서는 크롭 경로를 구현하지 않고
    "원본 전체에서 탐지 후 결과만 필터" 경로만 둔다. 크롭 버전을 보려면
    clipseg_qc_test.ipynb 를 쓴다(그쪽에 crop_to_garment 경로가 그대로 있다).
    """
    if CROP_BEFORE_DETECT:
        raise NotImplementedError(
            "이 노트북은 크롭 경로를 구현하지 않습니다(실측 회귀로 기본 OFF). "
            "크롭 실험은 clipseg_qc_test.ipynb 를 쓰세요."
        )

    raw = detect_all(pil_image)
    benigns = benign_instances(pil_image) if SUPPRESS_BENIGN else []
    garment_mask = garment_mask_full(pil_image) if RESTRICT_TO_GARMENT else None

    after_garment, off_garment = filter_by_garment(raw, garment_mask, GARMENT_OVERLAP_MIN)
    final, on_benign = suppress_benign(after_garment, benigns, BENIGN_CONTAINMENT)

    trace = {
        "raw": len(raw),
        "off_garment": len(off_garment),
        "on_benign": len(on_benign),
        "final": len(final),
        "garment_found": garment_mask is not None,
    }
    return renumber(final), trace


print(f"이미지 {len(IMAGE_PATHS)}장 CLIPSeg 탐지 시작 (하이브리드 1단)")
print()

clipseg_by_image = {}
traces_by_image = {}
for _path in IMAGE_PATHS:
    _final, _trace = inspect_image(load_image(_path))
    clipseg_by_image[_path] = _final
    traces_by_image[_path] = _trace
    print(
        f"  {_path.name:<40} raw {_trace['raw']:>3} "
        f"-> 배경 제외 -{_trace['off_garment']:<3} "
        f"-> 주름·그림자 억제 -{_trace['on_benign']:<3} "
        f"=> {_trace['final']:>3}건"
    )

_total = sum(len(v) for v in clipseg_by_image.values())
print()
print(f"정제 대상 인스턴스 총 {_total}건")

## 2. 1단 결과 요약 (정제 전 baseline)

이 표는 `clipseg_qc_test.ipynb` §4 와 **같은 값이 나와야 한다.** 다르면 공유 로직이 어긋난
것이므로 Tracker 를 붙이기 전에 원인을 찾아야 한다.

In [ ]:
import pandas as pd

_rows = []
for _path, _insts in clipseg_by_image.items():
    _t = traces_by_image[_path]
    _v, _peak = decide_verdict(_insts, VERDICT_THRESHOLDS)
    _row = {
        "image": _path.name,
        "raw": _t["raw"],
        "off_garment": -_t["off_garment"],
        "on_benign": -_t["on_benign"],
        "final": _t["final"],
        "garment_found": _t["garment_found"],
    }
    for _type in DEFECT_PROMPTS:
        _row[_type] = sum(1 for d in _insts if d["type"] == _type)
    _row["max_conf"] = round(_peak, 4)
    _row["verdict"] = _v
    _rows.append(_row)

baseline_df = pd.DataFrame(_rows)
print(baseline_df.to_string(index=False))
print()
print("⚠️ clipseg_qc_test.ipynb §4 표와 대조할 것 — 값이 같아야 공유 로직이 정상이다.")
baseline_df

## 3. SAM3Tracker 로드 — VRAM 동시 적재 실측

LAB-46 §4 는 **CLIPSeg(~150MB) + SAM3Tracker(fp16 ~1.7GB) ≈ 1.85GB** 로 추정하고
"RTX 4070 8GB 에서 둘 다 여유 있게 들어간다, 자원은 결정 기준이 아니다"라고 결론냈다.
**그 추정을 여기서 처음 실측한다.**

### 왜 Tracker 를 새로 임포트하는가

`Sam3Model`(PCS)의 `forward` 는 마스크·박스 재입력 기반 정제를 지원하지 않는다. 박스 프롬프트로
마스크를 얻는 건 `Sam3TrackerModel`(PVS, SAM2 계열 API) 전용이다. 같은 `facebook/sam3`
체크포인트지만 **서로 다른 `nn.Module` 이라 가중치를 공유하지 않는다**(그 델타를 아래에서 잰다).

### OOM 폴백

동시 적재가 8GB 를 넘으면 CLIPSeg 를 내리고 Tracker 단독으로 재시도한다. §1 탐지가 이미
끝났으므로 **인스턴스는 잃지 않는다.** 단 그 경우 아래 델타는 "동시 점유 추가분"이 아니게 되니
`VRAM_FALLBACK_USED` 플래그로 표시해 결론에서 혼동하지 않게 한다.

In [ ]:
import gc

from transformers import Sam3TrackerModel, Sam3TrackerProcessor

VRAM_FALLBACK_USED = False
tracker_model = None
tracker_processor = None


def load_tracker():
    proc = Sam3TrackerProcessor.from_pretrained(SAM3_MODEL_ID)
    mdl = Sam3TrackerModel.from_pretrained(SAM3_MODEL_ID, dtype=TRACKER_DTYPE).to(DEVICE).eval()
    return mdl, proc


_before, _ = vram_report("Tracker 로드 직전")
try:
    tracker_model, tracker_processor = load_tracker()
except torch.cuda.OutOfMemoryError as exc:
    print()
    print("[CUDA OOM] CLIPSeg 와 Tracker 동시 적재가 이 GPU 에서 실패했습니다.")
    print(f"  원인: {exc}")
    print("  -> CLIPSeg 를 내리고 Tracker 단독으로 재시도합니다(§1 인스턴스는 CPU 에 있어 안전).")
    del clipseg_model
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    VRAM_FALLBACK_USED = True
    _before, _ = vram_report("CLIPSeg 해제 후")
    tracker_model, tracker_processor = load_tracker()
except Exception as exc:
    print(f"[Tracker 로드 실패] {type(exc).__name__}: {exc}")
    print("  facebook/sam3 는 gated repo 입니다 — HF 약관 동의·승인 여부를 확인하세요.")
    raise

VRAM_BOTH, VRAM_PEAK = vram_report("Tracker 로드 직후")
VRAM_TRACKER_DELTA = VRAM_BOTH - _before

print()
print(f"Tracker 로드로 늘어난 allocated : {VRAM_TRACKER_DELTA:+.1f} MB "
      f"({VRAM_TRACKER_DELTA / 1024:+.2f} GB)")
if VRAM_FALLBACK_USED:
    print("  ⚠️ 폴백 경로 — CLIPSeg 를 내린 뒤 측정한 값이라 '동시 점유'가 아니다.")
    print("     LAB-46 의 1.85GB 추정과 직접 비교하면 안 된다.")
else:
    print(f"동시 적재 합계                  : {VRAM_BOTH / 1024:.2f} GB")
    print("  LAB-46 추정치                 : 1.85 GB")
    print(f"  차이                          : {VRAM_BOTH / 1024 - 1.85:+.2f} GB")

## 4. CLIPSeg 박스 -> Tracker 정제 마스크

`sam3_qc_test.ipynb` §9 의 변환 패턴을 그대로 쓰되, **입력 박스를 SAM3 PCS 대신 CLIPSeg
인스턴스에서 가져온다** — 이게 이 노트북의 핵심이자 지금까지 코드로 존재하지 않았던 부분이다.

```python
inputs = tracker_processor(images=pil, input_boxes=[boxes], return_tensors="pt")
outputs = tracker_model(**inputs, multimask_output=False)
refined = tracker_processor.post_process_masks(
    outputs.pred_masks.detach().cpu().float(), inputs["original_sizes"].cpu()
)[0]
```

### dtype 주의

Tracker 를 fp16 으로 올렸는데 processor 는 fp32 `pixel_values` 를 준다. 그대로 넣으면 첫 conv
에서 `Input type (float) and bias type (c10::Half) should be the same` 로 죽는다 —
`_to_tracker_device()` 가 `pixel_values` 만 캐스팅한다.

### 박스를 나눠 넣는 이유

박스가 많은 이미지에서 한 번에 넣으면 VRAM 이 튄다. 추론 중 OOM 이 나면 그 이미지 결과를 통째로
잃으므로 `REFINE_BATCH_SIZE` 단위로 쪼개 호출한다(대신 vision 인코더가 chunk 마다 다시 돈다).

In [ ]:
def _to_tracker_device(inputs):
    """텐서를 DEVICE 로 옮기고 pixel_values 만 Tracker dtype 에 맞춘다."""
    inputs = inputs.to(DEVICE)
    if "pixel_values" in inputs and inputs["pixel_values"].dtype.is_floating_point:
        inputs["pixel_values"] = inputs["pixel_values"].to(TRACKER_DTYPE)
    return inputs


def refine_boxes(pil_image, boxes):
    """박스 리스트 -> (정제 마스크 리스트, tracker 자체 iou_score 리스트).

    boxes 는 xyxy 절대 픽셀. REFINE_BATCH_SIZE 단위로 쪼개 호출하고 결과를 이어 붙인다.
    """
    masks_out, scores_out = [], []
    for start in range(0, len(boxes), REFINE_BATCH_SIZE):
        chunk = [[float(v) for v in b] for b in boxes[start : start + REFINE_BATCH_SIZE]]
        inputs = _to_tracker_device(
            tracker_processor(images=pil_image, input_boxes=[chunk], return_tensors="pt")
        )
        with torch.no_grad():
            outputs = tracker_model(**inputs, multimask_output=False)
        refined = tracker_processor.post_process_masks(
            outputs.pred_masks.detach().cpu().float(),
            inputs["original_sizes"].cpu(),
        )[0]
        iou_scores = outputs.iou_scores.detach().float().cpu().numpy().reshape(len(chunk), -1)
        for i in range(len(chunk)):
            masks_out.append(refined[i, 0].numpy().astype(np.uint8))
            scores_out.append(float(iou_scores[i][0]))
    return masks_out, scores_out


def classify_refinement(iou, px_ratio):
    """정제 결과를 3분류. ⚠️ 임계치는 근거 없는 초기값이다(설정 셀 참고).

    - TARGET_CHANGED : 같은 대상이라고 보기 어렵다(class-agnostic 함정) -> 정제로 못 쓴다
    - TIGHTENED      : 경계가 좁아졌다 -> 정제 이득의 실체
    - SIMILAR        : 거의 안 변했다 -> 정제해도 얻는 게 없다
    """
    if iou < REFINE_IOU_MIN or px_ratio > REFINE_PX_RATIO_MAX:
        return "TARGET_CHANGED"
    if px_ratio < REFINE_TIGHTEN_MAX:
        return "TIGHTENED"
    return "SIMILAR"


print("Tracker 정제 시작")
print()

refined_by_image = {}  # path -> list of dict(정제 결과 + 지표)
for _path in IMAGE_PATHS:
    _insts = clipseg_by_image[_path]
    if not _insts:
        refined_by_image[_path] = []
        print(f"  {_path.name:<40} CLIPSeg 인스턴스 0건 — 건너뜀")
        continue

    _img = load_image(_path)
    try:
        _masks, _scores = refine_boxes(_img, [d["bbox"] for d in _insts])
    except torch.cuda.OutOfMemoryError as exc:
        print(f"  {_path.name:<40} [CUDA OOM] 추론 실패: {exc}")
        print("    REFINE_BATCH_SIZE 를 줄이거나 커널을 재시작하세요.")
        refined_by_image[_path] = []
        continue

    _rows = []
    for _inst, _mask, _score in zip(_insts, _masks, _scores):
        _px_before = int(_inst["mask"].sum())
        _px_after = int(_mask.sum())
        _ratio = (_px_after / _px_before) if _px_before else 0.0
        _iou = mask_iou(_inst["mask"], _mask)
        _rows.append(
            {
                "no": _inst["no"],
                "type": _inst["type"],
                "confidence": _inst["confidence"],
                "bbox": _inst["bbox"],
                "px_before": _px_before,
                "px_after": _px_after,
                "px_ratio": round(_ratio, 4),
                "iou": round(_iou, 4),
                "tracker_iou_score": round(_score, 4),
                "refine_class": classify_refinement(_iou, _ratio),
                "mask_before": _inst["mask"],
                "mask_after": _mask,
            }
        )
    refined_by_image[_path] = _rows
    _counts = {c: sum(1 for r in _rows if r["refine_class"] == c) for c in
               ("TIGHTENED", "SIMILAR", "TARGET_CHANGED")}
    print(
        f"  {_path.name:<40} {len(_rows):>3}건 정제  "
        f"타이트 {_counts['TIGHTENED']:<3} 유사 {_counts['SIMILAR']:<3} "
        f"대상변경 {_counts['TARGET_CHANGED']:<3}"
    )

## 5. 정제 전/후 지표 — 이 노트북의 핵심 산출물

007 계획이 실측하려던 LAB-46 점검 기준 중 **3(마스크 경계 품질)** 에 해당한다.

**읽는 법**

| 지표 | 의미 |
| --- | --- |
| `px_ratio` | 정제 후/전 마스크 픽셀 비. **1 미만이면 경계가 좁아진 것** = 기대했던 이득 |
| `iou` | 정제 전/후 마스크 IoU. 높아야 "같은 대상의 경계만 달라진" 것이다 |
| `tracker_iou_score` | Tracker 가 **스스로 예측한** 마스크 품질. 위 `iou` 와 다른 값이다 |
| `refine_class` | TIGHTENED(이득) / SIMILAR(무의미) / TARGET_CHANGED(함정에 빠짐) |

**핵심 판단**: `TARGET_CHANGED` 비율이 높으면 이 하이브리드는 지금 형태로 쓸 수 없다 —
Tracker 가 하자 대신 원단 표면을 잡고 있다는 뜻이고, LAB-46 이 "저위험 증분"이라고 본 전제가
깨진다. 반대로 `TIGHTENED` 가 많으면 LAB-46 의 추론이 실측으로 확인되는 것이다.

In [ ]:
_rows = []
for _path, _items in refined_by_image.items():
    for _r in _items:
        _rows.append(
            {
                "image": _path.name,
                "no": _r["no"],
                "type": _r["type"],
                "conf": _r["confidence"],
                "px_before": _r["px_before"],
                "px_after": _r["px_after"],
                "px_ratio": _r["px_ratio"],
                "iou": _r["iou"],
                "trk_score": _r["tracker_iou_score"],
                "refine_class": _r["refine_class"],
            }
        )

if not _rows:
    print("정제된 인스턴스가 없습니다 — §1 에서 CLIPSeg 가 아무것도 못 찾았거나 §4 가 실패했습니다.")
    refine_df = None
else:
    refine_df = pd.DataFrame(_rows)
    print(refine_df.to_string(index=False))
    print()
    print("=" * 78)
    print("분류 집계")
    print("=" * 78)
    _total = len(refine_df)
    for _cls in ("TIGHTENED", "SIMILAR", "TARGET_CHANGED"):
        _n = int((refine_df["refine_class"] == _cls).sum())
        print(f"  {_cls:<16} {_n:>3}건  ({_n / _total:.1%})")
    print()
    print(f"  px_ratio 중앙값 : {refine_df['px_ratio'].median():.3f}  (1 미만이면 전반적으로 좁아짐)")
    print(f"  iou      중앙값 : {refine_df['iou'].median():.3f}")
    print(f"  trk_score 중앙값: {refine_df['trk_score'].median():.3f}")
    print()
    _changed = refine_df[refine_df["refine_class"] == "TARGET_CHANGED"]
    if len(_changed):
        print(f"⚠️ 대상변경 의심 {len(_changed)}건 — class-agnostic 함정 후보:")
        print(_changed[["image", "no", "type", "px_ratio", "iou"]].to_string(index=False))
    print()
    print("⚠️ 이 분류는 근거 없는 임계치(IoU<{}, px_ratio>{}, tighten<{})에 의존한다.".format(
        REFINE_IOU_MIN, REFINE_PX_RATIO_MAX, REFINE_TIGHTEN_MAX))
    print("   결론을 쓸 때 임계치를 함께 적을 것.")

refine_df

### 5-1. 눈으로 확인 — 정제 전/후 나란히

지표만으로는 "타이트해진 것"과 "대상이 바뀐 것"이 구별되지 않는 경우가 있다. confidence 상위
인스턴스를 확대해 윤곽선을 나란히 본다. 빨강=CLIPSeg 원본 경계, 초록=Tracker 정제 경계.

In [ ]:
import cv2
import matplotlib.pyplot as plt

print(f"matplotlib 한글 폰트: {use_korean_font()}")


def draw_two_contours(pil_image, mask_before, mask_after, thickness=2):
    """원본 경계(빨강) + 정제 경계(초록)를 같은 캔버스에 얹는다. RGB ndarray 반환."""
    canvas = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
    for mask, color in ((mask_before, (0, 0, 255)), (mask_after, (0, 255, 0))):
        contours, _ = cv2.findContours(
            mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE
        )
        cv2.drawContours(canvas, contours, -1, color, thickness)
    return cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)


def crop_around(img_array, bbox, pad_ratio=0.6):
    """bbox 를 중심으로 pad_ratio 만큼 넓혀 crop."""
    h, w = img_array.shape[:2]
    x1, y1, x2, y2 = (int(v) for v in bbox)
    pad_x = int((x2 - x1) * pad_ratio) + 4
    pad_y = int((y2 - y1) * pad_ratio) + 4
    return img_array[max(0, y1 - pad_y) : min(h, y2 + pad_y), max(0, x1 - pad_x) : min(w, x2 + pad_x)]


_all = [(p, r) for p, items in refined_by_image.items() for r in items]
_all.sort(key=lambda pair: pair[1]["confidence"], reverse=True)
_topk = _all[:TOPK_VISUALIZE]

if not _topk:
    print("정제 결과가 없어 시각화를 건너뜁니다.")
else:
    print(f"전체 {len(_all)}건 중 confidence 상위 {len(_topk)}건을 확대합니다.")
    print("빨강 = CLIPSeg 원본 / 초록 = Tracker 정제")
    fig, axes = plt.subplots(len(_topk), 2, figsize=(10, 5.0 * len(_topk)), squeeze=False)
    for _row, (_path, _r) in enumerate(_topk):
        _img = load_image(_path)
        _overlay = draw_two_contours(_img, _r["mask_before"], _r["mask_after"])
        _label = KOREAN_LABELS.get(_r["type"], _r["type"])

        axes[_row][0].imshow(crop_around(np.array(_img), _r["bbox"]))
        axes[_row][0].set_title(f"{_path.name} #{_r['no']} {_label} — 원본 crop", fontsize=10)
        axes[_row][0].axis("off")

        axes[_row][1].imshow(crop_around(_overlay, _r["bbox"]))
        axes[_row][1].set_title(
            f"{_r['refine_class']}  px {_r['px_before']}->{_r['px_after']} "
            f"(x{_r['px_ratio']:.2f})\nIoU={_r['iou']:.3f}  trk_score={_r['tracker_iou_score']:.3f}",
            fontsize=10,
        )
        axes[_row][1].axis("off")
    plt.tight_layout()
    plt.show()

    print()
    print("해석 주의: Tracker 는 class-agnostic 이라 박스 안의 '물체'를 분할한다.")
    print("초록 경계가 하자가 아니라 옷 전체/원단 표면을 감싸고 있으면 '정제'가 아니라 '대상 변경'이다.")

### 5-2. 전체 이미지 오버레이 저장

각 이미지의 모든 인스턴스에 대해 원본/정제 경계를 함께 얹은 그림을 저장한다.
발표자료·노션 문서에 붙일 근거 그림이다.

In [ ]:
for _path, _items in refined_by_image.items():
    if not _items:
        continue
    _img = load_image(_path)
    _canvas = cv2.cvtColor(np.array(_img), cv2.COLOR_RGB2BGR)
    for _r in _items:
        for _mask, _color in ((_r["mask_before"], (0, 0, 255)), (_r["mask_after"], (0, 255, 0))):
            _contours, _ = cv2.findContours(
                _mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE
            )
            cv2.drawContours(_canvas, _contours, -1, _color, 2)
    _out = cv2.cvtColor(_canvas, cv2.COLOR_BGR2RGB)
    Image.fromarray(_out).save(OUTPUT_DIR / f"{_path.stem}_hybrid_contours.png")

    # 번호 붙은 박스 그림도 함께 — 세 노트북이 같은 함수로 그려 눈으로 비교된다.
    _boxed = draw_numbered_boxes(_img, clipseg_by_image[_path])
    Image.fromarray(_boxed).save(OUTPUT_DIR / f"{_path.stem}_hybrid_boxes.png")

print(f"오버레이 저장 완료 -> {OUTPUT_DIR.resolve()}")

## 6. SAM3 단독과 비교

`sam3-condition-check/outputs/<파일명>_qc.json` 을 읽어 한 표에 놓는다. 없으면 건너뛴다.

### ⚠️ 이 비교로 말할 수 있는 것과 없는 것

**말할 수 있다** — 두 아키텍처가 **같은 곳을 짚는가**(bbox·개수·판정 수준).

**말할 수 없다** — **마스크 경계 품질의 직접 비교.** SAM3 노트북이 저장하는 JSON 은
`to_serializable()` 을 거쳐 **mask 를 버리고** bbox/confidence/type 만 남긴다. SAM3 PCS 의
고해상도 마스크와 이 하이브리드의 정제 마스크를 픽셀 단위로 비교하려면 SAM3 를 다시 돌려야
하는데, 그러면 CLIPSeg·Tracker·PCS 세 모델을 동시에 올려야 해서 이번 스코프 밖이다.

또한 **하이브리드의 탐지 결과는 CLIPSeg 단독과 동일하다**(§0 개념 설명 참고). 그래서 아래
개수·판정 비교는 사실상 "CLIPSeg vs SAM3" 이고, 이미
`clipseg_qc_test.ipynb` §7 이 같은 비교를 하고 있다. 여기서는 하이브리드 기준으로 한 번 더
확인하는 정도로만 본다.

In [ ]:
import json

_sam3_rows = []
_missing = []
for _path in IMAGE_PATHS:
    _f = SAM3_OUTPUT_DIR / f"{_path.stem}_qc.json"
    if not _f.exists():
        _missing.append(_path.name)
        continue
    _d = json.loads(_f.read_text(encoding="utf-8"))
    _insts = _d.get("instances", [])
    _row = {"image": _path.name, "sam3_total": len(_insts), "sam3_verdict": _d.get("verdict")}
    for _type in DEFECT_PROMPTS:
        _row[f"sam3_{_type}"] = sum(1 for i in _insts if i["type"] == _type)
    _row["sam3_max_conf"] = max((i["confidence"] for i in _insts), default=0.0)
    _sam3_rows.append(_row)

if _missing:
    print(f"[알림] SAM3 결과가 없는 이미지 {len(_missing)}장: {_missing}")
    print("       sam3-condition-check/sam3_qc_test.ipynb 를 끝까지 실행하면 생성됩니다.")

if not _sam3_rows:
    print("SAM3 결과가 하나도 없어 비교를 건너뜁니다.")
    compare_df = None
else:
    _hyb = baseline_df[["image", "final", "max_conf", "verdict"]].rename(
        columns={
            "final": "hybrid_total",
            "max_conf": "hybrid_max_conf",
            "verdict": "hybrid_verdict",
        }
    )
    compare_df = _hyb.merge(pd.DataFrame(_sam3_rows), on="image", how="inner")
    compare_df = compare_df[
        [
            "image",
            "hybrid_total",
            "sam3_total",
            "hybrid_max_conf",
            "sam3_max_conf",
            "hybrid_verdict",
            "sam3_verdict",
        ]
    ]
    print(compare_df.to_string(index=False))
    print()
    _agree = (compare_df["hybrid_verdict"] == compare_df["sam3_verdict"]).sum()
    print(f"판정 일치: {_agree}/{len(compare_df)}장")
    print()
    print("재확인: hybrid_total 은 CLIPSeg 탐지 결과다 — Tracker 는 개수를 바꾸지 않는다.")

compare_df

## 7. 결과 저장

`outputs/<파일명>_hybrid_qc.json` — 인스턴스별 정제 지표 포함.
`outputs/hybrid_qc_summary.json` — 이미지별 요약 + VRAM 실측 + 사용한 임계치.

임계치를 함께 저장하는 이유: `refine_class` 분류가 근거 없는 초기값에 의존하므로, 나중에
임계치를 바꿔 재해석할 때 **어떤 값으로 뽑은 결과인지** 알 수 있어야 한다.

In [ ]:
_summary_rows = []
for _path, _items in refined_by_image.items():
    _insts = clipseg_by_image[_path]
    _v, _ = decide_verdict(_insts, VERDICT_THRESHOLDS)

    _metrics = [
        {k: v for k, v in _r.items() if k not in ("mask_before", "mask_after")} for _r in _items
    ]
    _payload = {
        "image": _path.name,
        "architecture": "clipseg+sam3tracker",
        "detector": {"backend": "clipseg", "model_id": CLIPSEG_MODEL_ID, "dtype": "float32"},
        "refiner": {"backend": "sam3tracker", "model_id": SAM3_MODEL_ID, "dtype": "float16"},
        "prompt_set": ACTIVE_PROMPT_SET,
        "verdict": _v,
        "instances": to_serializable(_insts),
        "refinement": _metrics,
    }
    (OUTPUT_DIR / f"{_path.stem}_hybrid_qc.json").write_text(
        json.dumps(_payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    _counts = {c: sum(1 for r in _items if r["refine_class"] == c) for c in
               ("TIGHTENED", "SIMILAR", "TARGET_CHANGED")}
    _summary_rows.append(
        {
            "image": _path.name,
            "verdict": _v,
            "instances": len(_insts),
            "refined": len(_items),
            **_counts,
        }
    )

_summary = {
    "architecture": "clipseg+sam3tracker",
    "prompt_set": ACTIVE_PROMPT_SET,
    "thresholds": {
        "clipseg_prob": PROB_THRESHOLD,
        "clipseg_garment_prob": GARMENT_PROB_THRESHOLD,
        "refine_iou_min": REFINE_IOU_MIN,
        "refine_px_ratio_max": REFINE_PX_RATIO_MAX,
        "refine_tighten_max": REFINE_TIGHTEN_MAX,
        "note": "refine_* 는 근거 없는 초기값이다 — 라벨링 검증셋 확보 후 스윕 필요",
    },
    "vram_mb": {
        "clipseg_only": round(VRAM_CLIPSEG_ONLY, 1),
        "both_models": round(VRAM_BOTH, 1),
        "tracker_delta": round(VRAM_TRACKER_DELTA, 1),
        "peak": round(VRAM_PEAK, 1),
        "fallback_used": VRAM_FALLBACK_USED,
        "lab46_estimate_gb": 1.85,
    },
    "images": _summary_rows,
}
(OUTPUT_DIR / "hybrid_qc_summary.json").write_text(
    json.dumps(_summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(f"저장 완료 -> {OUTPUT_DIR.resolve()}")
print()
print(pd.DataFrame(_summary_rows).to_string(index=False))
print()
if VRAM_FALLBACK_USED:
    print("⚠️ VRAM 은 폴백 경로로 측정됐다 — 동시 적재값이 아니다.")
else:
    print(f"VRAM 동시 적재 {VRAM_BOTH / 1024:.2f} GB (LAB-46 추정 1.85 GB)")

---

## 결론 (실행 후 채울 것)

007 계획이 실측하겠다고 한 LAB-46 점검 기준 1·3·4 에 대응한다.

| # | 항목 | 근거 셀 | 결과 |
| --- | --- | --- | --- |
| 1 | **CLIPSeg+Tracker 동시 적재 VRAM** — LAB-46 추정 1.85GB 가 맞았는가 | §3, §7 | |
| 2 | Tracker 로드가 추가로 먹은 VRAM (같은 체크포인트인데 공유하지 않음을 확인) | §3 | |
| 3 | **TIGHTENED 비율** — 정제로 경계가 실제로 좁아진 사례가 있는가 | §5 | |
| 4 | **TARGET_CHANGED 비율** — class-agnostic 함정에 빠진 사례가 얼마나 되는가 | §5, §5-1 | |
| 5 | `px_ratio` / `iou` 중앙값 — 전반적 경향 | §5 | |
| 6 | `tracker_iou_score` 와 실제 `iou` 가 일치하는가 (Tracker 자체 신뢰도가 쓸 만한가) | §5 | |
| 7 | 1단 baseline 이 `clipseg_qc_test.ipynb` §4 와 일치하는가 (공유 로직 정상성) | §2 | |
| 8 | 하이브리드 vs SAM3 단독 판정 일치율 | §6 | |

### 이 노트북이 답하지 못하는 것 (과대 해석 방지)

- **정량 정확도가 아니다.** 라벨링된 정답셋이 없어 precision/recall 을 계산할 수 없다.
  `TIGHTENED` 가 많다고 해서 "더 정확하다"가 증명되는 게 아니다 — "경계가 좁아졌다"까지만이다.
  좁아진 경계가 실제 하자 영역과 맞는지는 육안 판단(§5-1)에 의존한다.
- **`refine_class` 는 임계치에 의존한다.** IoU 0.35 / px_ratio 2.0 / tighten 0.8 은 근거 없는
  초기값이다. 임계치를 바꾸면 분류 비율이 바뀐다 — 결론에 임계치를 반드시 함께 적을 것.
- **SAM3 PCS 마스크와의 픽셀 단위 비교가 없다.** SAM3 저장 JSON 에 mask 가 없어서다(§6 참고).
  "정제된 CLIPSeg 마스크가 SAM3 PCS 마스크만큼 좋은가"는 여전히 미답이다.
- **탐지 성능은 CLIPSeg 그대로다.** LAB-12 가 지목한 찢어짐·훼손 약점은 이 하이브리드로
  개선되지 않는다 — Tracker 는 탐지에 기여하지 않는다. 그 약점은 별도 대응이 필요하다.

### 다음 단계 후보

- [ ] LAB-46 문서 "다음 단계" 체크박스 중 **"CLIPSeg 박스 → SAM Tracker 정제 조합을 실제로
      코드화해서 성능 재확인"** 을 이 노트북 결과로 채운다
- [ ] 소규모 라벨링 검증셋(15~20장) 구축 후 두 아키텍처 정량 비교 — LAB-46 열린 질문 1번
- [ ] `sam3_qc_test.ipynb` §8(마스크 경계 품질) 결론 채우기 — 아직 비어 있다
- [ ] `TARGET_CHANGED` 가 많다면: 박스 대신 **마스크 프롬프트**나 포인트 프롬프트로 바꿔 재시도